In [1]:
import pandas as pd
import numpy as np
import re, pickle, gzip, os

In [2]:
def convert_sample_names(x):
    x = re.sub(r'^noncanonical_', '', x)
    x = re.sub(r'_fdp$', '', x)
    return x

cutoffs = pd.read_csv('noncanonical_groupwalk_qvalue_thresholds.csv')
cutoffs = cutoffs[cutoffs.group_type=='PTM']
cutoffs['sample'] = cutoffs['sample'].apply(convert_sample_names)
cutoffs.q_sel_final = np.round(cutoffs.q_sel_final / 100, 4)
# cutoffs.q_sel_final = cutoffs.apply(lambda x: 0.1 if x["fail_sel_cutoff"] else np.round(x["fdp_cutoff_q_max"]/100,3), axis=1)
cutoffs.reset_index(drop=True, inplace=True)
cutoffs.tail()

,sample,subset,dataset,database,search_type,group_type,approach,min_fdp,max_score_FDP_cutoff,fdp_cutoff,fdp_cutoff_q_min,fdp_cutoff_q_max,fail_sel_cutoff,q_sel_final,fdp_final
43,AM17,noncanonical,PXD005833,open,OpenSearch,PTM,groupwalk,0.0,2.438435,0.938914,0.286909,0.521821,False,0.0052,0.938914
44,AM18,noncanonical,PXD005833,open,OpenSearch,PTM,groupwalk,0.0,2.080098,0.922722,0.504383,1.430344,False,0.0143,0.922722
45,AM19,noncanonical,PXD005833,open,OpenSearch,PTM,groupwalk,0.0,2.333031,0.820513,0.471094,1.381733,False,0.0138,0.820513
46,AM20,noncanonical,PXD005833,open,OpenSearch,PTM,groupwalk,0.0,2.160399,0.919540,0.479056,1.278025,False,0.0128,0.919540
47,AM21,noncanonical,PXD005833,open,OpenSearch,PTM,groupwalk,0.0,1.661764,0.806452,0.732798,1.136510,False,0.0114,0.806452


In [5]:
cutoffs.database = 'openprot'
cutoffs.subset   = 'NonCanonical'

In [3]:
# Read the files in each sample
def read_files_in_each_sample():
    working_folder = "C:/Users/Enrico/OneDrive - UGent/run-ionbot"
    PXDs = [
        "PXD002057.v0.11.4",
        "PXD005833.v0.11.4",
        "PXD014258.v0.11.4"
    ]
    data = []
    for dataset_name in PXDs:
        for exp in os.scandir(os.path.join(working_folder, dataset_name, f'{dataset_name}-canon')):
            if not os.path.isdir(exp.path):
                continue
            tmp = re.sub(r'-canon$', '', exp.name)
            tmp = tmp.split('.')[0]
            tmp2 = pd.read_csv(os.path.join(exp,'group-walk-output.csv'),
                                              usecols=['spectrum_file']).spectrum_file
            tmp2 = [_.split('.')[0] for _ in tmp2.unique()]
            # print(exp, tmp, tmp2)
            data.append([exp, tmp, tmp2])
    return pd.DataFrame(data, columns=['path','sample','files'])

file_lists = read_files_in_each_sample()

In [4]:
file_lists = file_lists.explode('files')
cutoffs = file_lists.merge(cutoffs, on='sample')
cutoffs

,path,sample,files,subset,dataset,database,search_type,group_type,approach,min_fdp,max_score_FDP_cutoff,fdp_cutoff,fdp_cutoff_q_min,fdp_cutoff_q_max,fail_sel_cutoff,q_sel_final,fdp_final
0,<DirEntry '130327_o2_01_hu_C1_2hr.mgf.gzip'>,130327_o2_01_hu_C1_2hr,130327_o2_01_hu_C1_2hr,noncanonical,PXD002057,open,ClosedSearch,PTM,groupwalk,22.222222,NaN,NaN,NaN,NaN,True,0.0010,22.222222
1,<DirEntry '130327_o2_01_hu_C1_2hr.mgf.gzip'>,130327_o2_01_hu_C1_2hr,130327_o2_01_hu_C1_2hr,noncanonical,PXD002057,open,OpenSearch,PTM,groupwalk,0.000000,1.454358,0.761974,0.276985,7.871326,True,0.0010,0.000000
2,<DirEntry '130327_o2_02_hu_P1_2hr.mgf.gzip'>,130327_o2_02_hu_P1_2hr,130327_o2_02_hu_P1_2hr,noncanonical,PXD002057,open,ClosedSearch,PTM,groupwalk,14.574899,NaN,NaN,NaN,NaN,True,0.0010,14.574899
3,<DirEntry '130327_o2_02_hu_P1_2hr.mgf.gzip'>,130327_o2_02_hu_P1_2hr,130327_o2_02_hu_P1_2hr,noncanonical,PXD002057,open,OpenSearch,PTM,groupwalk,0.000000,2.370897,0.833333,0.236642,0.766191,False,0.0077,0.833333
4,<DirEntry '130327_o2_03_hu_C2_2hr.mgf.gzip'>,130327_o2_03_hu_C2_2hr,130327_o2_03_hu_C2_2hr,noncanonical,PXD002057,open,ClosedSearch,PTM,groupwalk,0.000000,1.375027,0.757576,2.090909,3.034352,False,0.0303,0.757576
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
67,<DirEntry 'SampleHela'>,SampleHela,ESC-HF-SampleHela5,noncanonical,PXD014258,open,OpenSearch,PTM,groupwalk,0.000000,1.431492,0.942481,0.856419,1.066362,False,0.0107,0.942481
68,<DirEntry 'SampleHela'>,SampleHela,ESC-HF-SampleHela2,noncanonical,PXD014258,open,ClosedSearch,PTM,groupwalk,0.000000,1.874731,0.909091,0.801867,1.266253,False,0.0127,0.909091
69,<DirEntry 'SampleHela'>,SampleHela,ESC-HF-SampleHela2,noncanonical,PXD014258,open,OpenSearch,PTM,groupwalk,0.000000,1.431492,0.942481,0.856419,1.066362,False,0.0107,0.942481
70,<DirEntry 'SampleHela'>,SampleHela,ESC-HF-SampleHela3,noncanonical,PXD014258,open,ClosedSearch,PTM,groupwalk,0.000000,1.874731,0.909091,0.801867,1.266253,False,0.0127,0.909091


In [6]:
for _ in ['sample','dataset','subset','database','search_type','group_type','approach','files']:
    print(_, len(set(cutoffs[_])), set(cutoffs[_]))

sample 24 {'130327_o2_04_hu_P2_2hr', 'AM9', '130327_o2_01_hu_C1_2hr', 'AM20', '130327_o2_03_hu_C2_2hr', '130327_o2_02_hu_P1_2hr', 'AM17', 'AM8', 'AM12', '130327_o2_06_hu_P3_2hr', 'AM11', 'AM21', 'AM16', 'AM7', '130327_o2_05_hu_C3_2hr', 'AM10', 'AM15', 'SampleHela', 'AM13', 'AM14', 'AM19', 'Sample-MCF', 'Sample-BT474', 'AM18'}
dataset 3 {'PXD002057', 'PXD005833', 'PXD014258'}
subset 1 {'NonCanonical'}
database 1 {'openprot'}
search_type 2 {'ClosedSearch', 'OpenSearch'}
group_type 1 {'PTM'}
approach 1 {'groupwalk'}
files 36 {'130327_o2_04_hu_P2_2hr', 'AM9', 'ESC-HF-SampleHela4', '130327_o2_01_hu_C1_2hr', 'AM20', '130327_o2_03_hu_C2_2hr', 'ESC-HF-Sample-BT474_1', '130327_o2_02_hu_P1_2hr', 'AM17', 'AM8', 'ESC-HF-Sample-MCF3', 'AM12', 'ESC-HF-Sample-BT474_4', '130327_o2_06_hu_P3_2hr', 'AM11', 'AM21', 'ESC-HF-Sample-BT474_2', 'AM16', 'AM7', '130327_o2_05_hu_C3_2hr', 'ESC-HF-Sample-MCF5', 'AM10', 'ESC-HF-Sample-MCF4', 'ESC-HF-Sample-MCF1', 'ESC-HF-SampleHela2', 'AM15', 'ESC-HF-Sample-MCF2', '

In [7]:
cutoffs2 = cutoffs.set_index(['files','subset','search_type']).to_dict()['q_sel_final']

In [8]:
with gzip.open('custom-groupwalk-cutoffs.gz','wb') as outfile:
    pickle.dump(cutoffs2, outfile)

In [9]:
with gzip.open('custom-groupwalk-cutoffs.gz','rb') as infile:
    b = pickle.load(infile)

In [10]:
cutoffs2 == b

True